In [ ]:
# 1. 安裝套件
!pip install gradio opencv-python-headless matplotlib numpy pillow -q

# 2. 載入套件與撰寫邏輯
import gradio as gr
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math

def process_all(input_img, view_mode, kernel_preset, custom_k_str, normalize_k, detector_choice):
    if input_img is None:
        return "請上傳圖片", None, None, "請上傳圖片", None, "請上傳圖片"

    # 影像資訊解析
    height, width, channels = input_img.shape
    file_size_kb = (input_img.nbytes) / 1024.0
    info_text = f"• 解析度：{width} × {height} 像素\n• 色彩通道：{channels} (RGB)\n• 記憶體占用：{file_size_kb:.2f} KB"

    # Tab 1: 色彩空間與直方圖
    fig_hist, ax = plt.subplots(figsize=(6, 3))
    if view_mode == "原始 RGB":
        disp_img = input_img.copy()
        for i, col in enumerate(['red', 'green', 'blue']):
            hist = cv2.calcHist([input_img], [i], None, [256], [0, 256])
            ax.plot(hist, color=col)
    elif view_mode == "RGB 通道拆解":
        r, g, b = input_img[:,:,0], input_img[:,:,1], input_img[:,:,2]
        disp_img = np.hstack((r, g, b))
        ax.hist(r.ravel(), 256, [0,256], color='red', alpha=0.5)
        ax.hist(g.ravel(), 256, [0,256], color='green', alpha=0.5)
        ax.hist(b.ravel(), 256, [0,256], color='blue', alpha=0.5)
    else: # YUV
        img_bgr = cv2.cvtColor(input_img, cv2.COLOR_RGB2BGR)
        yuv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YUV)
        disp_img = np.hstack((yuv[:,:,0], yuv[:,:,1], yuv[:,:,2]))
        ax.plot(cv2.calcHist([yuv], [0], None, [256], [0, 256]), color='black', label='Y')
        ax.plot(cv2.calcHist([yuv], [1], None, [256], [0, 256]), color='purple', label='U')
        ax.plot(cv2.calcHist([yuv], [2], None, [256], [0, 256]), color='orange', label='V')

    ax.set_xlim([0, 256])
    ax.grid(True, linestyle='--', alpha=0.5)
    fig_hist.tight_layout()

    # Tab 2: 卷積與 PSNR
    if kernel_preset == "邊緣檢測 (Laplacian)":
        k = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
    elif kernel_preset == "高斯平滑":
        k = np.array([[1/16, 2/16, 1/16], [2/16, 4/16, 2/16], [1/16, 2/16, 1/16]], dtype=np.float32)
    elif kernel_preset == "圖像銳化":
        k = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
    else:
        try:
            rows = custom_k_str.strip().split('\n')
            k = np.array([[float(x) for x in r.split(',')] for r in rows], dtype=np.float32)
        except:
            k = np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=np.float32)

    if normalize_k and np.sum(k) != 0:
        k = k / np.sum(k)

    filtered_img = cv2.filter2D(input_img, -1, k)
    filtered_img = np.clip(filtered_img, 0, 255).astype(np.uint8)

    mse = np.mean((input_img.astype(np.float64) - filtered_img.astype(np.float64)) ** 2)
    psnr_val = "∞ dB (完全相同)" if mse == 0 else f"{10 * math.log10((255.0 ** 2) / mse):.2f} dB"

    # Tab 3: 人臉偵測
    gray = cv2.cvtColor(input_img, cv2.COLOR_RGB2GRAY)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, 1.1, 5)

    face_img = input_img.copy()
    for i, (x, y, w, h) in enumerate(faces):
        cv2.rectangle(face_img, (x, y), (x+w, y+h), (0, 255, 0), 3)
        cv2.putText(face_img, f"Face #{i+1}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    face_summary = f"🎯 共偵測到 {len(faces)} 個人臉"

    return info_text, disp_img, fig_hist, psnr_val, filtered_img, face_img, face_summary

# 3. 建立 Gradio UI 介面 (直接在 Colab 內嵌入展示，不需外聯)
with gr.Blocks(title="AI 影像處理工具") as demo:
    gr.Markdown("# 📷 AI 輔助互動式影像處理與人臉分析工具")
    gr.Markdown("### 學號：C113196111 | 姓名：張紜瑄 | 期中專題實作成果")

    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="numpy", label="請在此上傳影像檔案")
            info_output = gr.Textbox(label="1. 影像中繼資訊解析", interactive=False)
            btn_run = gr.Button("🚀 開始執行分析處理", variant="primary")

        with gr.Column(scale=2):
            with gr.Tab("🎨 1 & 2. 色彩空間與直方圖"):
                view_mode = gr.Radio(["原始 RGB", "RGB 通道拆解", "YUV 色彩空間"], value="原始 RGB", label="檢視模式")
                img_color_out = gr.Image(label="色彩空間轉換結果")
                plot_hist_out = gr.Plot(label="動態直方圖分析")

            with gr.Tab("🔲 3. 自訂卷積濾波與 PSNR"):
                preset_k = gr.Dropdown(["邊緣檢測 (Laplacian)", "高斯平滑", "圖像銳化", "自訂 (使用下方文字框)"], value="邊緣檢測 (Laplacian)", label="濾波範本")
                custom_k = gr.Textbox(value="0,1,0\n1,-4,1\n0,1,0", label="自訂 3x3/5x5 矩陣 (用逗號隔開數字，換行隔開列)")
                norm_k = gr.Checkbox(label="開啟權重正規化 (Normalization)", value=False)
                psnr_out = gr.Textbox(label="📊 峰值信噪比 (PSNR) 計算結果", interactive=False)
                img_filter_out = gr.Image(label="空間域濾波結果圖")

            with gr.Tab("👤 4. 人臉偵測模組"):
                face_summary_out = gr.Textbox(label="偵測統計結果", interactive=False)
                img_face_out = gr.Image(label="人臉邊界框標註畫面")

    btn_run.click(
        fn=process_all,
        inputs=[img_input, view_mode, preset_k, custom_k, norm_k],
        outputs=[info_output, img_color_out, plot_hist_out, psnr_out, img_filter_out, img_face_out, face_summary_out]
    )

demo.launch(share=True, debug=True)

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1379: UserWarning: Expected 6 arguments for function <function process_all at 0x798d37435c60>, received 5.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1383: UserWarning: Expected at least 6 arguments for function <function process_all at 0x798d37435c60>, received 5.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://89b23dba99c0e063dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/helpers.py:1193: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
/usr/local/lib/python3.13/dist-packages/gradio/helpers.py:1193: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
/usr/local/lib/python3.13/dist-packages/gradio/helpers.py:1193: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
/usr/local/lib/python3.13/dist-packages/gradio/helpers.py:1193: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
/tmp/ipykernel_2125/997662796.py:30: MatplotlibDeprecationWarning: Passing the range parameter of hist() positionally is deprecated since Matplotlib 3.9; the parameter will become keyword-only in 3.11.
  ax.hist(r.ravel(), 256, [0,256], color='red', alpha=0.5)
/tmp/ipykernel_2125/997662796.py:31